In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .getOrCreate()
products_data = [
    ("P01", "Laptop", "Electronics", 60000),
    ("P02", "Monitor", "Electronics", 14000),
    ("P03", "Headphones", "Accessories", 3500),
    ("P04", "Printer", "Office", 17000),
    ("P05", "Router", "Networking", 4500),
    ("P06", "Tablet", "Electronics", 24000)
]
products_df = spark.createDataFrame(
    products_data,
    ["product_id", "product_name", "category", "list_price"]
)
stores_data = [
    ("S01", "Hyderabad", "South"),
    ("S02", "Bengaluru", "South"),
    ("S03", "Mumbai", "West"),
    ("S04", "Delhi", "North")
]
stores_df = spark.createDataFrame(
    stores_data,
    ["store_id", "city", "region"]
)
products_df.show()
stores_df.show()
sales_df = (
    spark.range(1, 100001)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "order_date",
        F.date_add(
            F.to_date(F.lit("2026-01-01")),
            (F.col("order_id") % 180).cast("int")
        )
    )
    .withColumn(
        "product_id",
        F.concat(
            F.lit("P"),
            F.lpad(
                ((F.col("order_id") % 6) + 1).cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "store_id",
        F.concat(
            F.lit("S"),
            F.lpad(
                ((F.floor(F.col("order_id") / 3) % 4) + 1)
                .cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "quantity",
        ((F.col("order_id") % 8) + 1).cast("int")
    )
    .withColumn(
        "discount_pct",
        F.when(
            F.col("order_id") % 19 == 0,
            F.lit(None).cast("int")
        ).otherwise(
            ((F.floor(F.col("order_id") / 5) % 4) * 5)
            .cast("int")
        )
    )
    .withColumn(
        "order_status",
        F.when(F.col("order_id") % 11 == 0, "Cancelled")
         .when(F.col("order_id") % 7 == 0, "Returned")
         .otherwise("Completed")
    )

    .withColumn(
        "payment_method",
        F.when(F.col("order_id") % 3 == 0, "UPI")
         .when(F.col("order_id") % 3 == 1, "Card")
         .otherwise("Cash")
    )

    .withColumn(
        "salesperson_id",
        F.when(
            F.col("order_id") % 17 == 0,
            F.lit(None).cast("string")
        ).otherwise(
            F.concat(
                F.lit("EMP"),
                F.lpad(
                    ((F.col("order_id") % 12) + 1).cast("string"),
                    2, "0"
                )
            )
        )
    )

    .join(
        products_df.select(
            "product_id",
            F.col("list_price").alias("unit_price")
        ),
        "product_id",
        "left"
    )

    .select(
        "order_id", "order_date", "store_id",
        "product_id", "quantity", "unit_price",
        "discount_pct", "order_status",
        "payment_method", "salesperson_id"
    )
)

print("Total Records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10, truncate=False)


+----------+------------+-----------+----------+
|product_id|product_name|   category|list_price|
+----------+------------+-----------+----------+
|       P01|      Laptop|Electronics|     60000|
|       P02|     Monitor|Electronics|     14000|
|       P03|  Headphones|Accessories|      3500|
|       P04|     Printer|     Office|     17000|
|       P05|      Router| Networking|      4500|
|       P06|      Tablet|Electronics|     24000|
+----------+------------+-----------+----------+

+--------+---------+------+
|store_id|     city|region|
+--------+---------+------+
|     S01|Hyderabad| South|
|     S02|Bengaluru| South|
|     S03|   Mumbai|  West|
|     S04|    Delhi| North|
+--------+---------+------+

Total Records: 100000
root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_p

**Practice Exercises**

**Part A | DataFrame transformations**

1. Select order_id, product_id, quantity, and unit_price from the sales DataFrame.

In [3]:
sales_df.select("order_id", "product_id", "quantity", "unit_price").show(10)

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|       3|       P04|       4|     17000|
|       9|       P04|       2|     17000|
|       2|       P03|       3|      3500|
|       8|       P03|       1|      3500|
|       4|       P05|       5|      4500|
|      10|       P05|       3|      4500|
|       5|       P06|       6|     24000|
|      11|       P06|       4|     24000|
|       1|       P02|       2|     14000|
|       7|       P02|       8|     14000|
+--------+----------+--------+----------+
only showing top 10 rows


2. Filter all completed transactions where quantity is greater than 5.

In [4]:
sales_df.filter((F.col("order_status") == "Completed") & (F.col("quantity") > 5)).show(10)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      38|2026-02-08|     S01|       P03|       7|      3500|        NULL|   Completed|          Cash|         EMP03|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|
|      23|2026-01-24|     S04|       P06|       8|     24000|           0|   Completed|          Cash|         EMP12|
|      29|2026-01-30|     S02|       P06|       6|     2

3. Create a column named gross_revenue by multiplying quantity and unit_price.

In [5]:
sales_rev_df = sales_df.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
sales_rev_df.show(10)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|        68000|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|        34000|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|        10500|
|       8|2026-01-09|     S03|       P03|       1|      3500|           5|   Completed|          Cash|         EMP09|         3500|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|  

4. Rename order_status to transaction_status, then drop the payment_method column.

In [6]:
rename_order_status_df = sales_df.withColumnRenamed("order_status", "transaction_status").drop("payment_method")
rename_order_status_df.show(10)
rename_order_status_df.printSchema()

+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|         Completed|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|         Completed|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|         Completed|         EMP03|
|       8|2026-01-09|     S03|       P03|       1|      3500|           5|         Completed|         EMP09|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|         Completed|         EMP05|
|      10|2026-01-11|     S04|       P05|       3|      4500|          10|         Completed|         EMP11|
|       5|2026-01-0

**Part B | Aggregations and joins**

5. Calculate the total quantity sold for each product.

In [9]:
sales_df.groupBy("product_id").agg(F.sum("quantity").alias("total_quantity")).orderBy("product_id").show()

+----------+--------------+
|product_id|total_quantity|
+----------+--------------+
|       P01|         66668|
|       P02|         83336|
|       P03|         66667|
|       P04|         83334|
|       P05|         66665|
|       P06|         83330|
+----------+--------------+



6. Join sales with product master data and calculate total gross revenue by product category.

In [10]:
sales_rev_df.join(products_df, "product_id", "inner").groupBy("category").agg(F.sum("gross_revenue").alias("total_gross_revenue")).orderBy(F.desc("total_gross_revenue")).show()

+-----------+-------------------+
|   category|total_gross_revenue|
+-----------+-------------------+
|Electronics|         7166704000|
|     Office|         1416678000|
| Networking|          299992500|
|Accessories|          233334500|
+-----------+-------------------+



7. Join sales with store master data and calculate total transactions for each city.

In [11]:
sales_df.join(stores_df, "store_id", "inner").groupBy("city").agg(F.count("order_id").alias("total_transactions")).orderBy(F.desc("total_transactions")).show()

+---------+------------------+
|     city|total_transactions|
+---------+------------------+
|Bengaluru|             25001|
|Hyderabad|             25001|
|   Mumbai|             24999|
|    Delhi|             24999|
+---------+------------------+



8. Join all three DataFrames and find the top five products by gross revenue for completed orders.

In [12]:
sales_rev_df.filter(F.col("order_status") == "Completed").join(products_df, "product_id").join(stores_df, "store_id").groupBy("product_id","product_name").agg(F.sum("gross_revenue").alias("total_gross_revenue")).orderBy(F.desc("total_gross_revenue")).limit(5).show()

+----------+------------+-------------------+
|product_id|product_name|total_gross_revenue|
+----------+------------+-------------------+
|       P01|      Laptop|         3116940000|
|       P06|      Tablet|         1558416000|
|       P04|     Printer|         1103912000|
|       P02|     Monitor|          909076000|
|       P05|      Router|          233766000|
+----------+------------+-------------------+



**Part C | Null handling and conditions**

9. Count null values in discount_pct and salesperson_id.

In [13]:
sales_df.select(
    F.sum(F.when(F.col("discount_pct").isNull(), 1).otherwise(0)).alias("null_discount_pct"),
    F.sum(F.when(F.col("salesperson_id").isNull(), 1).otherwise(0)).alias("null_salesperson_id")
).show()

+-----------------+-------------------+
|null_discount_pct|null_salesperson_id|
+-----------------+-------------------+
|             5263|               5882|
+-----------------+-------------------+



10. Replace null discounts with zero using na.fill().

In [14]:
replaced_null_df = sales_df.na.fill({"discount_pct": 0})
replaced_null_df.show(10)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|   Completed|           UPI|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|   Completed|          Cash|         EMP03|
|       8|2026-01-09|     S03|       P03|       1|      3500|           5|   Completed|          Cash|         EMP09|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|   Completed|          Card|         EMP05|
|      10|2026-01-11|     S04|       P05|       3|      

11. Remove records where salesperson_id is null using na.drop().

In [15]:
dropped_df = sales_df.na.drop(subset=["salesperson_id"])
print("Before:", sales_df.count(), "| After:", dropped_df.count())

Before: 100000 | After: 94118


12. Create order_size using when() and otherwise(): Small for quantity 1-3, Medium for 4-6, and Large
for 7 or more.

In [16]:
sized_df = sales_df.withColumn("order_size",
                               F.when(F.col("quantity").between(1,3), "Small")
                                .when(F.col("quantity").between(4,6), "Medium")
                                .otherwise("Large")
                               )
sized_df.groupBy("order_size").count().show()

+----------+-----+
|order_size|count|
+----------+-----+
|    Medium|37500|
|     Small|37500|
|     Large|25000|
+----------+-----+



**Part D | Window and date functions**

13. Rank products by total gross revenue within each category using dense_rank().

In [17]:
products_rev_df = sales_rev_df.join(products_df, "product_id").groupBy("category", "product_id", "product_name").agg(F.sum("gross_revenue").alias("total_gross_revenue"))
w_cat = Window.partitionBy("category").orderBy(F.desc("total_gross_revenue"))
products_rev_df.withColumn("rank", F.dense_rank().over(w_cat)).orderBy("category", "rank").show()

+-----------+----------+------------+-------------------+----+
|   category|product_id|product_name|total_gross_revenue|rank|
+-----------+----------+------------+-------------------+----+
|Accessories|       P03|  Headphones|          233334500|   1|
|Electronics|       P01|      Laptop|         4000080000|   1|
|Electronics|       P06|      Tablet|         1999920000|   2|
|Electronics|       P02|     Monitor|         1166704000|   3|
| Networking|       P05|      Router|          299992500|   1|
|     Office|       P04|     Printer|         1416678000|   1|
+-----------+----------+------------+-------------------+----+



14. Calculate running gross revenue by store, ordered by date and order ID.

In [18]:
w_run = (Window.partitionBy("store_id").orderBy("order_date", "order_id")
         .rowsBetween(Window.unboundedPreceding, Window.currentRow))
sales_rev_df.withColumn("running_gross_revenue", F.sum("gross_revenue").over(w_run)).orderBy("store_id", "order_date", "order_id").show()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+---------------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|gross_revenue|running_gross_revenue|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+-------------+---------------------+
|     180|2026-01-01|     S01|       P01|       5|     60000|           0|   Completed|           UPI|         EMP01|       300000|               300000|
|     360|2026-01-01|     S01|       P01|       1|     60000|           0|   Completed|           UPI|         EMP01|        60000|               360000|
|     540|2026-01-01|     S01|       P01|       5|     60000|           0|   Completed|           UPI|         EMP01|       300000|               660000|
|     720|2026-01-01|     S01|       P01|       1|     60000|           0|  

15. Extract the month from order_date and calculate monthly sales revenue for completed orders.

In [19]:
sales_rev_df.filter(F.col("order_status") == "Completed").withColumn("month", F.month("order_date")).groupBy("month").agg(F.sum("gross_revenue").alias("monthly_sales_revenue")).orderBy("month").show()

+-----+---------------------+
|month|monthly_sales_revenue|
+-----+---------------------+
|    1|           1278728500|
|    2|           1027585500|
|    3|           1212087500|
|    4|           1218921000|
|    5|           1277560500|
|    6|           1089027500|
+-----+---------------------+



**Part E | Lazy execution**

16. Create three consecutive transformations: filter(), withColumn(), and select(). Add a Python
print() after each transformation. Trigger execution only at the end using show().

In [20]:
t1 = sales_df.filter(F.col("order_status") == "Completed")
print("filter() defined")

t2 = t1.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
print("withColumn() defined")

t3 = t2.select("order_id", "product_id", "quantity", "gross_revenue")
print("select() defined")

print("Now trigerring execution with show()...")
t3.show(5)

filter() defined
withColumn() defined
select() defined
Now trigerring execution with show()...
+--------+----------+--------+-------------+
|order_id|product_id|quantity|gross_revenue|
+--------+----------+--------+-------------+
|       3|       P04|       4|        68000|
|       2|       P03|       3|        10500|
|       4|       P05|       5|        22500|
|       5|       P06|       6|       144000|
|       1|       P02|       2|        28000|
+--------+----------+--------+-------------+
only showing top 5 rows


17. Create an aggregation by product_id. Use explain() to inspect the execution plan before calling
collect().

In [21]:
agg_df = sales_rev_df.groupBy("product_id").agg(F.sum("gross_revenue").alias("total_gross_revenue"))
agg_df.explain()
agg_df.collect()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#33], functions=[sum(gross_revenue#130L)])
   +- HashAggregate(keys=[product_id#33], functions=[partial_sum(gross_revenue#130L)])
      +- Project [product_id#33, (cast(quantity#35 as bigint) * unit_price#40L) AS gross_revenue#130L]
         +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
            :- Sort [product_id#33 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=5311]
            :     +- Project [concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, cast(((id#30L % 8) + 1) as int) AS quantity#35]
            :        +- Range (1, 100001, step=1, splits=2)
            +- Sort [product_id#0 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(product_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=5312]
                  +- Project [product_id#0, list_price#3L AS unit_price

[Row(product_id='P04', total_gross_revenue=1416678000),
 Row(product_id='P03', total_gross_revenue=233334500),
 Row(product_id='P05', total_gross_revenue=299992500),
 Row(product_id='P06', total_gross_revenue=1999920000),
 Row(product_id='P02', total_gross_revenue=1166704000),
 Row(product_id='P01', total_gross_revenue=4000080000)]

**Part F | Caching and persistence**

18. Filter completed orders, call cache(), and execute count() twice. Measure the execution time of
each action.

In [24]:
completed_df = sales_df.filter(F.col("order_status") == "Completed").cache()

start = time.time()
count1 = completed_df.count()
t_first = time.time() - start

start = time.time()
count2 = completed_df.count()
t_second = time.time() - start

print(f"Time taken for first count: {count1} rows in {t_first:.3f}s (computes + caches)")
print(f"Time taken for second count: {count2} rows in {t_second:.3f}s (served from cache)")
completed_df.unpersist()

Time taken for first count: 77923 rows in 3.588s (computes + caches)
Time taken for second count: 77923 rows in 1.086s (served from cache)


DataFrame[order_id: bigint, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: bigint, discount_pct: int, order_status: string, payment_method: string, salesperson_id: string]

19. Persist transactions with quantity greater than 5 using StorageLevel.MEMORY_AND_DISK. Execute
two actions, then release the persisted data using unpersist().

In [25]:
big_qty_df = sales_df.filter(F.col("quantity") > 5).persist(StorageLevel.MEMORY_AND_DISK)

print("Count:", big_qty_df.count())
print("Distinct products:", big_qty_df.select("product_id").distinct().count())
print("Storage level:", big_qty_df.storageLevel)

big_qty_df.unpersist()
print("Is cached after unpersist:", big_qty_df.is_cached)

Count: 37500
Distinct products: 6
Storage level: Disk Memory Serialized 1x Replicated
Is cached after unpersist: False


20. Cache an aggregated DataFrame, inspect its execution plan, and identify InMemoryTableScan or
InMemoryRelation.

In [27]:
cached_agg = sales_rev_df.groupBy("store_id").agg(F.sum("gross_revenue").alias("total_gross_revenue")).cache()
cached_agg.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [store_id#34, total_gross_revenue#2421L]
      +- InMemoryRelation [store_id#34, total_gross_revenue#2421L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=true
               +- == Final Plan ==
                  ResultQueryStage 4
                  +- *(4) HashAggregate(keys=[store_id#34], functions=[sum(gross_revenue#130L)])
                     +- ShuffleQueryStage 3
                        +- Exchange hashpartitioning(store_id#34, 200), ENSURE_REQUIREMENTS, [plan_id=6295]
                           +- *(3) HashAggregate(keys=[store_id#34], functions=[partial_sum(gross_revenue#130L)])
                              +- *(3) Project [store_id#34, (cast(quantity#35 as bigint) * unit_price#40L) AS gross_revenue#130L]
                                 +- *(3) BroadcastHashJoin [product_id#33], [product_id#0], LeftOuter, BuildRight, false
                         

**Part G | Repartition and coalesce**

21. Check the existing partition count, then repartition the sales DataFrame into 8 partitions.

In [28]:
print("Existing partitions:", sales_df.rdd.getNumPartitions())
repartitioned_df = sales_df.repartition(8)
print("New partitions:", repartitioned_df.rdd.getNumPartitions())

Existing partitions: 2
New partitions: 8


22. Display the number of records in each partition using spark_partition_id().

In [29]:
repartitioned_df.groupBy(F.spark_partition_id()).count().orderBy("spark_partition_id()").show()

+--------------------+-----+
|SPARK_PARTITION_ID()|count|
+--------------------+-----+
|                   0|12500|
|                   1|12500|
|                   2|12500|
|                   3|12500|
|                   4|12500|
|                   5|12500|
|                   6|12500|
|                   7|12500|
+--------------------+-----+



23. Repartition the sales DataFrame into 6 partitions based on store_id. Inspect which stores appear
in each partition.

In [31]:
sales_by_store = sales_df.repartition(6, "store_id")
sales_by_store.groupBy(F.spark_partition_id()).agg(F.collect_set("store_id")).orderBy("spark_partition_id()").show()

+--------------------+---------------------+
|SPARK_PARTITION_ID()|collect_set(store_id)|
+--------------------+---------------------+
|                   1|                [S03]|
|                   3|                [S02]|
|                   4|                [S01]|
|                   5|                [S04]|
+--------------------+---------------------+



24. Create a DataFrame with 12 partitions, then reduce it to 4 using coalesce(). Compare its
execution plan with repartition(4).

In [32]:
df12 = sales_df.repartition(12)
print("Partitions:", df12.rdd.getNumPartitions())

print("=== coalesce(4) plan ===")
df12.coalesce(4).explain()

print("=== repartition(4) plan ===")
df12.repartition(4).explain()

print("Coalesce partitions:", df12.coalesce(4).rdd.getNumPartitions())
print("Repartition partitions:", df12.repartition(4).rdd.getNumPartitions())

Partitions: 12
=== coalesce(4) plan ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 4
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=7329]
      +- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39]
         +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
            :- Sort [product_id#33 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=7323]
            :     +- Project [id#30L AS order_id#31L, date_add(2026-01-01, cast((id#30L % 180) as int)) AS order_date#32, concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, concat(S, lpad(cast(((FLOOR((cast(id#30L as double) / 3.0)) % 4) + 1) as string), 2, 0)) AS store_id#34, cast(((id#30L % 8) + 1) as int) AS quantity#35, CASE WHEN ((id#30L % 19) = 0) THEN null ELSE cast(((F

25. Save the sales data into 4 CSV part files using repartition(4) and write.csv().

In [34]:
import os
import glob

output_csv = "/content/output/sales_csv"
sales_df.repartition(4).write.mode("overwrite").option("header", True).csv(output_csv)

print(sorted(os.listdir(output_csv)))
print("CSV part files:", len(glob.glob(output_csv + "/part-*.csv")))

['._SUCCESS.crc', '.part-00000-296f6d0c-46cc-46e7-9f25-1f85873ee6c5-c000.csv.crc', '.part-00001-296f6d0c-46cc-46e7-9f25-1f85873ee6c5-c000.csv.crc', '.part-00002-296f6d0c-46cc-46e7-9f25-1f85873ee6c5-c000.csv.crc', '.part-00003-296f6d0c-46cc-46e7-9f25-1f85873ee6c5-c000.csv.crc', '_SUCCESS', 'part-00000-296f6d0c-46cc-46e7-9f25-1f85873ee6c5-c000.csv', 'part-00001-296f6d0c-46cc-46e7-9f25-1f85873ee6c5-c000.csv', 'part-00002-296f6d0c-46cc-46e7-9f25-1f85873ee6c5-c000.csv', 'part-00003-296f6d0c-46cc-46e7-9f25-1f85873ee6c5-c000.csv']
CSV part files: 4


**Final Challenge**

**End-to-end PySpark data-processing pipeline**

Build a single pipeline using sales_df, products_df, and stores_df. Complete the following steps:
1. Filter completed orders.
2. Replace null discounts with zero.
3. Join product and store details.
4. Calculate gross revenue and discounted revenue.
5. Group the data by region and product category.
6. Calculate total revenue and transaction count.
7. Rank categories by revenue within each region.
8. Cache the final aggregated DataFrame.
9. Display the results.
10. Write the output into 2 Parquet part files.

In [40]:
final_df = (
    sales_df
    # 1. filter completed orders only
    .filter(F.col("order_status") == "Completed")
    # 2. replace null discounts -> 0
    .na.fill({"discount_pct": 0})
    # 3. join product and store details
    .join(products_df.select("product_id", "product_name", "category"), "product_id")
    .join(stores_df, "store_id")
    # 4. calculate gross and discounted revenue
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .withColumn("discounted_revenue",
                F.col("gross_revenue") * (1 - F.col("discount_pct") / 100))
    # 5 + 6. group by region and category; calculate revenue and transaction count
    .groupBy("region", "category")
    .agg(
        F.round(F.sum("gross_revenue"), 2).alias("total_gross_revenue"),
        F.round(F.sum("discounted_revenue"), 2).alias("total_discounted_revenue"),
        F.count("order_id").alias("transaction_count")
    )
)

# 7. rank categories by revenue within each region
w_region = Window.partitionBy("region").orderBy(F.desc("total_discounted_revenue"))
final_df = final_df.withColumn("category_rank", F.dense_rank().over(w_region))

# 8. cache the final aggregated DataFrame
final_df = final_df.cache()

# 9. display results
final_df.orderBy("region", "category_rank").show()

# 10. write the output into 2 Parquet part files
output_parquet = "/content/output/final_parquet"
(final_df.repartition(2)
    .write.mode("overwrite")
    .parquet(output_parquet))

print(sorted(os.listdir(output_parquet)))
print("Parquet part files:", len(glob.glob(output_parquet + "/part-*.parquet")))

+------+-----------+-------------------+------------------------+-----------------+-------------+
|region|   category|total_gross_revenue|total_discounted_revenue|transaction_count|category_rank|
+------+-----------+-------------------+------------------------+-----------------+-------------+
| North|Electronics|          934848000|              8.551824E8|             6492|            1|
| North|     Office|          441388000|               4.12114E8|             6492|            2|
| North| Networking|          146097000|            1.35026775E8|             6494|            3|
| South|Electronics|         2156236000|             2.0242794E9|            19484|            1|
| South|     Office|          662524000|              6.060058E8|             6495|            2|
| South|Accessories|          113631000|            1.05018025E8|             6494|            3|
| South| Networking|           87669000|             8.2684125E7|             6494|            4|
|  West|Electronics|